# Feature engineering

Two candidates are tested here, both questions about which columns reach the model: a flag for
negative equity, and correlation-based selection for the linear branch. The 64 features are
already derived ratios, so hand-built combinations of them are unlikely to help a tree, which
finds combinations on its own. What can help is a state the current representation cannot
express.

Each candidate is stated as a hypothesis before it is run, with an expectation per branch and
the mechanism behind that expectation. Configurations are then compared fold by fold on the same
25 folds, and a null result is recorded exactly like a positive one. Generating twenty
candidates and keeping whichever scored best would fit the feature set to the cross-validation
noise, and the resulting number would not survive contact with the holdout.

Everything below runs on the training split alone — 4387 rows, 306 positives. The holdout stays
closed.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, QuantileTransformer

from src.config import (
    DATA_5YEAR_PATH,
    EXCLUDED_FEATURES,
    MISSING_INDICATOR_COLS,
    RANDOM_STATE,
)
from src.data import split_holdout
from src.features import CorrelationSelector, add_negative_equity_flag
from src.pipelines import make_linear_pipeline
from src.train import compare_pipelines

X_train, X_holdout, y_train, y_holdout = split_holdout(DATA_5YEAR_PATH)
feature_cols = [c for c in X_train.columns if c not in EXCLUDED_FEATURES]

# Both should sit at ~0.070. This is the only check that catches a forgotten stratify=.
print(y_train.mean(), y_holdout.mean())

0.0697515386368817 0.06971975393028025


In [2]:
def report(diff: np.ndarray) -> str:
    """Mean difference with twice its standard error."""
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    return f"{diff.mean():+.4f} ± {2 * se:.4f}"


def show(diff: dict[str, np.ndarray]) -> None:
    """Print one line per metric: paired difference and how often it held."""
    for name, d in diff.items():
        print(f"{name:15s} {report(d)}   folds in favour: {(d > 0).sum()} / {len(d)}")

## 1. Negative equity

Negative equity means liabilities exceed assets: the company is technically insolvent. The
candidate came out of the `Attr59` ECDF in the feature analysis, where 10% of bankrupts sat
below zero against 2.6% of survivors.

`Attr59 < 0` is nevertheless the wrong way to encode it, and the reason is the numerator rather
than any missing values. `Attr59` is long-term liabilities over equity, and the audit found no
long-term debt at all in 43% of the file, where the ratio is a legitimate zero whatever the sign
of equity. The condition would silently place those companies in the "equity not negative" group
without grounds. That is a blind spot, not a weak signal: a weak signal misclassifies and the
error can be measured, while here the feature carries no information about the state on those
rows at all.

Other columns that put equity against assets or liabilities — `Attr53`, `Attr8`, `Attr17` —
share a milder version of the same defect, since fixed assets and total liabilities can both be
zero. `Attr10`, equity over total assets, divides by a quantity that is strictly positive for
any operating company, so the sign of the ratio is the sign of equity on every row.

`Attr2 > 1` should be the same condition. By the balance-sheet identity
`equity = assets − liabilities`, dividing through by assets gives `Attr10 = 1 − Attr2`. Whether
the file honours that identity is an empirical question.

In [3]:
# equity = assets - liabilities implies Attr10 = 1 - Attr2, so the two thresholds
# should select the same rows. Whether they do is an empirical question about the file.
pd.crosstab(X_train["Attr2"] > 1, X_train["Attr10"] < 0)

Attr10,False,True
Attr2,,
False,4133,17
True,0,237


In [4]:
# The identity holds only approximately; size and direction of the gap.
gap = X_train["Attr10"] + X_train["Attr2"] - 1
print(gap.abs().median(), (gap.abs() > 0.01).mean())
print((gap > 1e-6).sum(), (gap < -1e-6).sum())

# Rows where the right-hand side overshoots — systematically smaller companies?
print(X_train["Attr29"].groupby(gap > 1e-6).median())

0.0 0.2762708000911785
210 1815
False    4.20305
True     3.52990
Name: Attr29, dtype: float64


**The two definitions almost agree.** 237 rows are flagged by both conditions, 17 by `Attr10`
alone, and none by `Attr2` alone. The empty cell is the informative one: the implication runs
strictly one way, `Attr2 > 1` entails `Attr10 < 0`, so `Attr10` is the wider criterion and picks
up 17 companies the leverage threshold misses.

The identity itself does not hold exactly. The gap is zero on half the rows and exceeds 0.01 on
28% of them, almost always in the negative direction — that is, `equity + liabilities` falls
short of assets. The most likely reading is that the dataset's total liabilities is a narrow
subset of the right-hand side of the balance sheet, with items such as provisions or deferred
income left out. The 210 rows where the sum overshoots cannot be explained that way, but their
median log-assets is 3.53 against 4.20 elsewhere: these are the smallest companies, where both
ratios divide by an assets figure close to zero and any rounding is amplified.

This settles the choice. A flag built on `Attr10` reads equity directly and does not depend on
how completely liabilities were counted, which the file has just shown to be unreliable.

In [5]:
# NaN < 0 evaluates to False, so rows without Attr10 land in the "equity not negative" group.
missing_equity = X_train["Attr10"].isna()
print(X_train.loc[missing_equity, ["Attr2", "Attr10"]])
print(X_train[missing_equity].isna().sum(axis=1))

      Attr2  Attr10
5820    NaN     NaN
4827    NaN     NaN
1764    NaN     NaN
5820    28
4827    41
1764    32
dtype: int64


**Three rows have no `Attr10` at all**, and each is missing 28, 41 and 32 of the 64 features.
`Attr2` is absent in the same rows, and its denominator is also total assets, so the shared
cause is a missing normalisation base rather than an unknown equity figure — these are filings
that were largely never populated.

`NaN < 0` evaluates to `False`, so these three companies fall into the "equity not negative"
group. Formally wrong, immaterial at 0.07% of the sample, and stated here rather than left
implicit. They are not dropped: a company with a sparse filing would still arrive for scoring in
production, and removing such rows from training would tune the model on a population it will
not meet.

### Hypothesis

A new feature is worth adding when it gives the model something it cannot extract from the
columns it already has. `Attr10` is already among the features, so the flag adds no information
about the company — only a different shape for the same information, a step at a single point.

**Boosting: no gain expected.** A tree searches thresholds along the axis by itself and will
find the split at zero if it pays. A pre-computed indicator duplicates work the algorithm
already does.

**Linear branch: gain expected only if risk is discontinuous at zero.** Logistic regression adds
a constant to the logit per unit of the feature, so it can express "lower equity, higher risk"
but not "risk jumps the moment equity turns negative". The `QuantileTransformer` ahead of it
does not change this: it is monotone, and rank-preserving transformations leave a step a step.
If insolvency is a qualitatively different state rather than the far end of a continuum, the
flag supplies what the model cannot build.

That condition is checkable before any model is fitted.

In [6]:
mask = X_train["Attr10"] < 0
print(mask.sum(), y_train[mask].mean(), y_train.mean())

# Zero has to be a bin boundary: qcut would place the first edge near the 25th percentile
# and bury the whole negative region inside one bin.
edges = [-0.2, -0.1, 0.0, 0.1, 0.2]
bins = pd.cut(X_train["Attr10"], bins=edges)
print(y_train.groupby(bins, observed=True).agg(["size", "mean"]))

254 0.29133858267716534 0.0697515386368817
              size      mean
Attr10                      
(-0.2, -0.1]    31  0.161290
(-0.1, 0.0]     68  0.176471
(0.0, 0.1]     168  0.202381
(0.1, 0.2]     235  0.102128


**The association is real; the discontinuity is not.** The flag covers 254 rows, 5.8% of the
sample, with a bankruptcy rate of 0.291 against 0.070 overall — 4.2 times higher, or 22
percentage points. About a quarter of all positives fall inside it.

The bands tell a different story from the group total. Moving across zero, the rate runs 0.161,
0.176, 0.202, 0.102, with 31 to 235 rows per band. There is no step at the boundary, and the
differences between adjacent bands are comparable to their standard errors — at 31 rows and a
rate near 0.16 the standard error is already about 0.07. Risk varies with the level of equity,
not with its sign.

**Expectation restated before the run: no gain in either branch.** The condition that would have
justified the flag for the linear model does not hold. The experiment is run anyway, because a
measured null belongs in the results table on the same footing as a measured gain.

### Measurement

The two pipelines differ in exactly one thing: the same model object and the same `feature_cols`
go into both, so anything else varying would make the comparison measure that instead. The flag
is built inside the pipeline rather than in a cell — nothing leaks either way, but a column
created in the notebook would not survive `joblib.dump`, and the rule would end up written twice.

`remainder="drop"` is spelled out rather than left to the default. `Attr21` is still present in
`X_train` — it was removed from `feature_cols`, not from the data — and `passthrough` would
carry it to the model, restoring roughly 0.08 PR-AUC that was deliberately given up.

In [7]:
model = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
)
base = make_linear_pipeline(model, feature_cols)

# Copy of the numeric branch built inside make_linear_pipeline: the factory keeps it local.
# Parameters must stay in sync with pipelines.py or the comparison stops being paired.
numeric_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "quantile",
            QuantileTransformer(
                n_quantiles=500,
                output_distribution="normal",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

prep_with_flag = ColumnTransformer(
    transformers=[
        ("flags", MissingIndicator(features="all"), MISSING_INDICATOR_COLS),
        ("flag_new", "passthrough", ["has_negative_equity"]),
        ("numeric_raw", numeric_pipe, feature_cols),
    ],
    remainder="drop",
)

pipe_with_flag = Pipeline(
    [
        ("features", FunctionTransformer(add_negative_equity_flag)),
        ("prep", prep_with_flag),
        ("model", model),
    ]
)
pipe_with_flag.set_output(transform="pandas");

In [8]:
# Positive difference means the pipeline with the flag is ahead.
show(compare_pipelines(pipe_with_flag, base, X_train, y_train))

pr_auc          -0.0015 ± 0.0022   folds in favour: 12 / 25
precision_at_k  +0.0000 ± 0.0099   folds in favour: 4 / 25


**No effect.** PR-AUC moves by −0.0015 against a threshold of 0.0022, and precision@top-3% by
0.0000 against 0.0099. Twelve folds out of 25 favour the flag on PR-AUC, which is a coin flip.
On precision the count is 4 out of 25 because most folds record a difference of exactly zero:
the top 26 companies in the queue are the same list with or without the flag.

The threshold is two standard errors of the per-fold differences, `2 * std / sqrt(25)`, not the
sum of two standard deviations. That is available because both pipelines ran on identical folds:
a fold that is hard by composition drags both configurations down, so subtracting removes the
shared noise and leaves the effect of the change. The differences here scatter by about 0.003
where the metric itself scatters by 0.042, an order of magnitude more sensitive. Independently
reported figures have no such pairing and have to be compared with the coarser rule.

The mechanism is the finding. Negative equity does predict bankruptcy — 0.291 against 0.070 —
but that information is already carried by the continuous `Attr10`, and the relationship turned
out to be monotone rather than stepped. Binarising a monotone relationship at an accounting
boundary only coarsens what the model could already use. The legal threshold of insolvency is
not a threshold of risk: a company at −2% of assets and one at +2% are in much the same
position.

## 2. Correlation-based selection

The 64 ratios are built from roughly fifteen raw balance-sheet quantities, so restated versions
of the same figure are unavoidable: 13 pairs sit above 0.99 Spearman and 34 above 0.95. For the
boosting branch this costs nothing — a tree takes one column out of a duplicated group and
ignores the rest. For the linear branch the design matrix is close to singular, and weight
inside such a group is distributed almost arbitrarily, so `coef_` cannot be read.

That framing sets the expectation. L2 regularisation is on by default in `LogisticRegression`
and already handles the instability: it spreads weight across the group instead of handing it to
one column, which keeps predictions stable. The hypothesis is therefore not that dropping
duplicates improves anything.

**Hypothesis (non-inferiority): a reduced feature set is no worse than the full 63.** Success is
the absence of an effect — the mean paired difference staying inside `2 * std / sqrt(25)`. What
would be bought with it is engineering, not accuracy: fewer columns to collect and validate in
production, less surface for data drift.

Two thresholds are examined rather than one. At 0.99 the columns are near-algebraic duplicates
and removal should be free by construction; 0.95 is already a judgement that two ratios measure
one factor. Both are reported, so that picking whichever scored better cannot masquerade as a
finding.

In [9]:
# Same greedy rule the selector implements, run here only to size the effect.
corr = X_train[feature_cols].corr(method="spearman").abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

for threshold in (0.99, 0.95):
    to_drop = [c for c in upper.columns if upper[c].gt(threshold).any()]
    print(threshold, len(to_drop), to_drop)

0.99 10 ['Attr8', 'Attr14', 'Attr17', 'Attr18', 'Attr26', 'Attr33', 'Attr52', 'Attr60', 'Attr61', 'Attr63']
0.95 19 ['Attr7', 'Attr8', 'Attr10', 'Attr11', 'Attr14', 'Attr17', 'Attr18', 'Attr23', 'Attr26', 'Attr31', 'Attr33', 'Attr47', 'Attr49', 'Attr52', 'Attr54', 'Attr60', 'Attr61', 'Attr62', 'Attr63']


**Ten columns go at 0.99, nineteen at 0.95**, leaving 53 and 44 of the 63. Both cuts are large
enough to be worth measuring.

The survivors are not the ones the pairwise list suggests. `Attr7` was expected to survive its
group with `Attr14` and `Attr18`, but it correlates with `Attr1` at 0.988 — net profit over
assets against EBIT over assets, the same ratio before and after interest and tax — and `Attr1`
stands further left. The rule drops a column when *any* column to its left crosses the
threshold, so transitive chains collapse further than the pair list implies.

Spot-checking the rest, the removals are economically sensible rather than accidental: `Attr10`
goes against `Attr8` at 0.989 (both read equity, denominators linked through the balance sheet),
`Attr26` against `Attr16` at 0.993 (the same ratio at gross and net profit), `Attr17` against
`Attr2` and `Attr60` against `Attr20` as mutually inverse pairs. Nothing was dropped on a
borderline 0.95 coincidence.

### Measurement

The selector sits inside `numeric_pipe`, ahead of imputation. Correlations are therefore computed
on observed values only: `pandas.corr` uses pairwise deletion, so each pair is measured on the
rows where both columns are present. Placing the selector after `SimpleImputer` would be worse
than that inconvenience — the missingness masks of `Attr45`/`Attr60` and of the fixed-asset block
coincide bitwise, so the same median would be written into the same rows of several columns at
once and inflate their correlation. Columns would then be dropped because of imputation rather
than because of duplication.

Being learnable is what forces the selector into the pipeline: the correlation matrix is a
statistic over rows, and computing it once on the full training split would let validation rows
decide which features exist. `fit` records the columns on the ~3510 training rows of each fold,
`transform` applies that list.

In [10]:
def make_reduced_pipeline(threshold: float) -> Pipeline:
    """The linear pipeline of the factory, with correlation selection ahead of imputation."""
    numeric_pipe = Pipeline(
        [
            ("corr", CorrelationSelector(threshold=threshold)),
            ("imputer", SimpleImputer(strategy="median")),
            (
                "quantile",
                QuantileTransformer(
                    n_quantiles=500,
                    output_distribution="normal",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    prep = ColumnTransformer(
        transformers=[
            ("flags", MissingIndicator(features="all"), MISSING_INDICATOR_COLS),
            ("numeric_raw", numeric_pipe, feature_cols),
        ],
        remainder="drop",
    )
    pipe = Pipeline([("prep", prep), ("model", model)])
    pipe.set_output(transform="pandas")
    return pipe


for threshold in (0.99, 0.95):
    print(f"threshold {threshold}")
    show(compare_pipelines(make_reduced_pipeline(threshold), base, X_train, y_train))

threshold 0.99
pr_auc          -0.0097 ± 0.0095   folds in favour: 10 / 25
precision_at_k  -0.0185 ± 0.0259   folds in favour: 5 / 25
threshold 0.95
pr_auc          -0.0285 ± 0.0135   folds in favour: 4 / 25
precision_at_k  -0.0385 ± 0.0270   folds in favour: 5 / 25


**Both cuts cost quality, and the cost scales with the cut.** At 0.99 PR-AUC falls by 0.0097
against a threshold of 0.0095 — marginal, but outside it, with 10 folds of 25 in favour. At 0.95
it falls by 0.0285 against 0.0135, with 4 folds of 25. precision@top-3% moves the same way,
−0.0185 and −0.0385, though only the second is outside its interval. Non-inferiority is rejected
at both thresholds.

The monotone ordering is what makes this readable as an effect rather than noise: a more
aggressive cut costs more. On precision the numbers are best translated back into the queue,
where the metric moves in steps of 1/26 per fold — the 0.95 cut loses one flagged bankrupt per
fold, and roughly two per month at the portfolio scale of 2000 counterparties and a queue of 60.

The expectation was wrong, and the interesting part is why. The argument for a null result was
that duplicated columns carry no information and that L2 already neutralises the instability they
cause. The first half does not hold even at 0.99: a rank correlation of 0.99 still leaves
variance that is not shared, and with 63 features against 4387 rows the model is nowhere near the
regime where extra columns hurt. There is no penalty here for keeping them, so the regularised
full set stays.

What the experiment does not license is the reverse claim that correlation-based selection is
useless in general. With far more features, or fewer rows, or a model without regularisation, the
trade would look different.

## Results

| candidate | branch | PR-AUC | precision@top-3% |
|---|---|---|---|
| `has_negative_equity` | linear | −0.0015 ± 0.0022 | +0.0000 ± 0.0099 |
| correlation cut at 0.99 (53 of 63) | linear | −0.0097 ± 0.0095 | −0.0185 ± 0.0259 |
| correlation cut at 0.95 (44 of 63) | linear | −0.0285 ± 0.0135 | −0.0385 ± 0.0270 |

Paired differences against the same pipeline without the candidate, 25 folds, positive means the
candidate is ahead. The interval is two standard errors of the differences; an effect is read as
real only when the mean exceeds it.

Neither candidate is adopted. The flag is rejected as measured-null, the correlation cuts as a
measured loss. The boosting branch was not run: the flag duplicates a split a tree finds by
itself, and correlated columns cost a tree nothing, so both expectations are recorded as argument
rather than measurement.